# Lab 2: Regular Expressions, Edit Distance, and Subword Unit Tokenization

In this lab, we will cover key topics, including regular expressions, edit distance, and subword unit tokenization using Byte Pair Encoding (BPE). This notebook contains exercises that students will need to complete in the code sections. The necessary files will also be provided with this notebook.


## Exercise 1: Basic Regular Expressions

Write regular expressions for the following languages:
1. The set of all alphabetic strings
2. The set of all lower-case alphabetic strings ending in a b
3. The set of all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b

In [11]:
import re

# 1. The set of all alphabetic strings
regex_1 = r'[a-zA-Z]*'

# 2. The set of all lower-case alphabetic strings ending in the letter b
regex_2 = r'[a-z]*b'

# 3. The set of all strings from the alphabet a,b such that each letter a is immediately preceded by and immediately followed by the letter b
regex_3 = r'(b+(ab+)*)?'

# Test the regular expressions
test_strings = ["", "Hello", "world", "abc", "b", "bb", "bab", "babab", "abab",
                "babbabbab", "bababab", "abba", "ba", "baab", "ab"]

for i, (regex,question) in enumerate([(regex_1,"all alphabetic strings"), (regex_2,"all lower-case alphabetic strings ending in a b"), (regex_3,"all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b")], 1):
    print(f"\nTesting regex {regex} which searches for {question}:")
    for test_str in test_strings:
        print(f"\"{test_str}\" matches: {bool(re.fullmatch(regex, test_str))}")


Testing regex [a-zA-Z]* which searches for all alphabetic strings:
"" matches: True
"Hello" matches: True
"world" matches: True
"abc" matches: True
"b" matches: True
"bb" matches: True
"bab" matches: True
"babab" matches: True
"abab" matches: True
"babbabbab" matches: True
"bababab" matches: True
"abba" matches: True
"ba" matches: True
"baab" matches: True
"ab" matches: True

Testing regex [a-z]*b which searches for all lower-case alphabetic strings ending in a b:
"" matches: False
"Hello" matches: False
"world" matches: False
"abc" matches: False
"b" matches: True
"bb" matches: True
"bab" matches: True
"babab" matches: True
"abab" matches: True
"babbabbab" matches: True
"bababab" matches: True
"abba" matches: False
"ba" matches: False
"baab" matches: True
"ab" matches: True

Testing regex (b+(ab+)*)? which searches for all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b:
"" matches: True
"Hello" matches: False
"world" matches:

## Exercise 2: Advanced Regular Expressions

Write regular expressions for the following languages. By "word", we mean an alphabetic string separated from other words by whitespace, any relevant punctuation, line breaks, and so forth.

1. The set of all strings with two consecutive repeated words (e.g., "Humbert Humbert" and "the the" but not "the bug" or "the big bug").
2. All strings that start at the beginning of the line with an integer and that end at the end of the line with a word.
3. All strings that have both the word grotto and the word raven in them (but not, e.g., words like grottos that merely contain the word grotto).
4. Write a pattern that captures the first word of an English sentence along with any ending punctuation if it is concatenated to it.

In [22]:
import re


regex_1 = r"\b([a-zA-Z]+)\W+\1\b"


regex_2 = r"^\d+(?!\.).*[a-zA-Z]+$"


regex_3 = r"(.*(\braven\b).*(\bgrotto\b).*)|(.*(\bgrotto\b).*(\braven\b).*)"


regex_4 = r"^\s*([a-zA-Z']+[.,?!:;]*)"


def run_tests(title, pattern, flags, cases, group=0):
    print(f"\nTesting {title}: {pattern}")
    for text, expected in cases:
        match = re.search(pattern, text, flags)
        found = match.group(group) if match else None
        status = "OK" if bool(match) == expected else "WRONG"
        print(f"  {text!r:55} -> {found!r}   [{status}]")


run_tests("regex 1 (repeated words)", regex_1, re.IGNORECASE, [
    ("Humbert Humbert is a character character", True),
    ("the the", True),
    ("The the cat", True),
    ("the bug", False),
    ("the big bug", False),
    ("the theory", False),       # prefix of the next word is NOT a repetition
])

run_tests("regex 2 (integer ... word)", regex_2, re.MULTILINE, [
    ("42 is the answer to everything", True),
    ("42 hello", True),
    ("The answer is 42", False),   # does not start with an integer
    ("42 is the answer 42", False),  # ends with a number, not a word
    ("42 is the answer.", False),  # ends with punctuation
    ("42", False),
    ("3.14 is pi", False),         # 3.14 is not an integer
])

run_tests("regex 3 (grotto + raven)", regex_3, re.IGNORECASE | re.DOTALL, [
    ("The raven visited the grotto", True),
    ("grotto raven", True),
    ("The grotto\nand the Raven", True),
    ("The raven visited the grottos", False),
    ("Only a raven", False),
    ("Only a grotto", False),
])

run_tests("regex 4 (first word + punctuation)", regex_4, 0, [
    ("Hello, world! How are you?", True),
    ("She was pleased.", True),
    ("Don't stop.", True),
    ("Wait... what?", True),
    ("  Yes!", True),
    ("42 is the answer", False),
], group=1)


Testing regex 1 (repeated words): \b([a-zA-Z]+)\W+\1\b
  'Humbert Humbert is a character character'              -> 'Humbert Humbert'   [OK]
  'the the'                                               -> 'the the'   [OK]
  'The the cat'                                           -> 'The the'   [OK]
  'the bug'                                               -> None   [OK]
  'the big bug'                                           -> None   [OK]
  'the theory'                                            -> None   [OK]

Testing regex 2 (integer ... word): ^\d+(?!\.).*[a-zA-Z]+$
  '42 is the answer to everything'                        -> '42 is the answer to everything'   [OK]
  '42 hello'                                              -> '42 hello'   [OK]
  'The answer is 42'                                      -> None   [OK]
  '42 is the answer 42'                                   -> None   [OK]
  '42 is the answer.'                                     -> None   [OK]
  '42'                  

## Exercise 3: Regular Expression for Employee Data Extraction

Using the provided HTML file containing employee information, write regular expressions to extract the following data: Full Name, Email Address, Phone Number, Personal Identification Code, Date of Birth, Website URL, City, Country, Job Title, Department.

Create a regex for each data point and save the extracted employee data in a JSON file.

In [ ]:
import html
import json
import re

# Read the HTML content from the file (the 'with' block closes the file properly)
with open("employee-data-html.html", "r", encoding="utf-8") as f:
    html_content = f.read()
    print(html_content)

    departement = re.compile(
        r'<div class="department-section" id="(.*?)">(.*?)</div>\s*(?=<div class="department-section"|<footer>|$)',
        re.S,
    )
    employees = re.compile(r'<div class="employee-card">(.*?)</div>', re.S)

    Name = re.compile(r'<p>Nom: <span class="firstname">(.*?)</span> <span class="lastname">(.*?)</span></p>')
    Email = re.compile(r"<p>Email: ([\w\.-]+@[\w\.-]+)</p>")
    Phone = re.compile(r"<p>Téléphone: (.*?)</p>")
    Id = re.compile(r"<p>Code d\'identification: (.*?)</p>")
    BirthDate = re.compile(r"<p>Date de naissance: (\d{4}-\d{2}-\d{2})</p>")
    Website = re.compile(r"<p>Site Web: (.*?)</p>")
    Ville = re.compile(r"<p>Ville: (.*?)</p>")

    result = []
    for dept_match in departement.finditer(html_content):
        dept_content = dept_match.group(2)
        for card_match in employees.finditer(dept_content):
            card_text = card_match.group(1)

            name_m = Name.search(card_text)
            email_m = Email.search(card_text)
            phone_m = Phone.search(card_text)
            id_m = Id.search(card_text)
            dob_m = BirthDate.search(card_text)
            site_m = Website.search(card_text)
            loc_m = Ville.search(card_text)

            name_val = (
                f"{name_m.group(1)} {name_m.group(2)}"
            )
            email_val = email_m.group(1)
            phone_val = phone_m.group(1)
            id_val = id_m.group(1)
            dob_val = dob_m.group(1)
            site_val = site_m.group(1)
            loc_val = loc_m.group(1)

            result.append(
                f"{name_val} with email {email_val} and with phone {phone_val} holding the id {id_val} he born {dob_val} his website is {site_val} living in {loc_val}\n"
            )

    for r in result:
        print(r)

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>TechMaghreb Solutions - Employee Directory</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 0; padding: 20px; background: #f5f5f5; }
        .header { text-align: center; background: #2c3e50; color: white; padding: 20px; margin-bottom: 30px; }
        .news-ticker { background: #34495e; color: white; padding: 10px; margin-bottom: 20px; }
        .employee-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 20px; }
        .employee-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        .department-section { margin-bottom: 30px; }
        footer { margin-top: 30px; text-align: center; padding: 20px; background: #2c3e50; color: white; }
    </style>
</head>
<body>
    <div class="header">
        <h1>TechMaghreb Solutions</h1>
        <p>Employee Directory - Updated June 2024</p>
    </div>

   

## Exercise 4: Create a Chatbot Inspired by ELIZA

Design a simple chatbot that mimics the style of ELIZA, an early AI program created in 1966 by Joseph Weizenbaum. ELIZA simulated a conversation with a psychotherapist by using pattern matching and simple responses.

In [21]:
import re

# Enhanced ELIZA-like chatbot
class ElizaChatbot:
    def __init__(self):
        self.response_rules = [
            (r'\bquit\b', 'Goodbye! It was nice talking to you.'),
            (r'\b(hello|hi)\b', 'Hello there! How are you feeling today?'),
            (r'i need (.*)', r'Why do you need \1?'),
        ]

    def eliza_response(self, user_input):
        user_input = user_input.strip()

        # Check for patterns with case insensitivity
        for pattern, response in self.response_rules:
            match = re.search(pattern, user_input, re.IGNORECASE)
            if match:
                # Use re.sub to inject capture groups (like \1) into the response
                return re.sub(pattern, response, user_input, flags=re.IGNORECASE)

        # Return a default response if no patterns matched
        return "I didn't quite understand that. Can you elaborate?"

# Example usage
if __name__ == "__main__":
    eliza = ElizaChatbot()
    print("ELIZA: Hello! I'm your virtual assistant. Type 'quit' to exit.")

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("ELIZA: Goodbye! It was nice talking to you.")
            break
        print("ELIZA:", eliza.eliza_response(user_input))

ELIZA: Hello! I'm your virtual assistant. Type 'quit' to exit.
ELIZA: Hello there! How are you feeling today?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: Why do you need chocolat?
ELIZA: Why do you need chocolat for fun?
ELIZA: Goodbye! It was nice talking to you.


## Exercise 5: Edit Distance

1. Compute the edit distance (using insertion cost 1, deletion cost 1, substitution cost 1) of "leda" to "deal". Show your work (using the edit distance grid).

2. Implement a minimum edit distance algorithm and use your hand-computed results to check your code.

In [1]:
# Function to calculate the minimum edit distance between two strings
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)

    # Create a 2D array to store distances
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Initialize the first row and column
    for i in range(m + 1):
        dp[i][0] = i  # Deletion
    for j in range(n + 1):
        dp[0][j] = j  # Insertion

    # Fill the rest of the grid
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # No operation needed
            else:
                dp[i][j] = min(
                    dp[i - 1][j] + 1,  # Deletion
                    dp[i][j - 1] + 1,  # Insertion
                    dp[i - 1][j - 1] + 1  # Substitution
                )

    # Return the minimum edit distance
    return dp[m][n]

# Function to print the edit distance grid
def print_edit_distance_grid(s1, s2, dp):
    print(f"Edit distance grid for '{s1}' to '{s2}':")
    print("   " + "  ".join(s2))
    for i in range(len(dp)):
        if i == 0:
            print(" ", end=" ")
        else:
            print(s1[i - 1], end=" ")
        for j in range(len(dp[i])):
            print(dp[i][j], end=" ")
        print()

# Compute and display the edit distance for "leda" to "deal"
source, target = "leda", "deal"

distance = edit_distance(source, target)
dp = [[0] * (len(target) + 1) for _ in range(len(source) + 1)]
for i in range(len(source) + 1):
    for j in range(len(target) + 1):
        if i == 0:
            dp[i][j] = j
        elif j == 0:
            dp[i][j] = i
        else:
            if source[i - 1] == target[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + 1)

print(f"Minimum edit distance between '{source}' and '{target}': {distance}")
print_edit_distance_grid(source, target, dp)

Minimum edit distance between 'leda' and 'deal': 3
Edit distance grid for 'leda' to 'deal':
   d  e  a  l
  0 1 2 3 4 
l 1 1 2 3 3 
e 2 2 1 2 3 
d 3 2 2 2 3 
a 4 3 3 2 3 


# Exercise 6: Spell Correction using NLTK

In this exercise, we'll implement spell correction functions using NLTK and the concepts of edit distance.

## Question 1:
Write a function that takes a misspelled word and returns the closest valid word from a given textual corpus using edit distance.

**Hint:** You can use the top 5000 most frequent words from an NLTK Gutenberg text such as 'austen-emma.txt'.

## Question 2:
Use the function you created in Question 1 to correct spelling errors in full sentences.

**Hint:** You can randomly introduce spelling errors into existing sentences from the Gutenberg Corpus available in NLTK by altering long words (those with more than 5 characters) through the insertion of a random character.


## Question 3:
Implement the following steps to evaluate the effectiveness of your spell correction:

1. **Select Sentences**: Choose a set of sentences from the Gutenberg Corpus available in NLTK.
2. **Introduce Spelling Errors**: Randomly alter some long words in each sentence to create spelling errors (through the insertion of a random character). Ensure that only words longer than 5 characters as before.
3. **Correct the Sentences**: Use the function from Question 1 to correct the misspelled words in each altered sentence.
4. **Print Results**: For each original sentence, print the altered sentence and the suggested corrections for the misspelled words.
5. **Calculate Precision**: Evaluate the accuracy of the corrections by calculating the precision of the spell correction method on the provided test set in the code section below.


In [9]:
import nltk
from nltk.corpus import gutenberg
from nltk.metrics.distance import edit_distance
from collections import Counter
import string
import random

# Download necessary NLTK datasets
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

# Question 1:
# 1. Build vocabulary of top 5,000 frequent words from austen-emma.txt
emma_words = [w.lower() for w in gutenberg.words('austen-emma.txt') if w.isalpha()]
vocab_counts = Counter(emma_words)
VOCABULARY = [word for word, _ in vocab_counts.most_common(5000)]
VOCAB_SET = set(VOCABULARY)

def correct_word(word, vocabulary=VOCABULARY):
    """
    Returns the closest valid word from the vocabulary using Levenshtein edit distance.
    If multiple words share the minimum distance, frequency rank acts as a tie-breaker.
    """
    word_lower = word.lower()
    
    # Direct match: 0 edit distance
    if word_lower in VOCAB_SET:
        return word_lower
    
    # Find the candidate word with minimum edit distance
    # Python's min() is stable: preserves most frequent order on ties
    best_candidate = min(vocabulary, key=lambda candidate: edit_distance(word_lower, candidate))
    return best_candidate

print(correct_word("exampel")) 

# Question 2:
def correct_sentence(sentence, vocabulary=VOCABULARY):
    """
    Tokenizes a sentence and replaces out-of-vocabulary words with 
    their closest match from the vocabulary.
    """
    tokens = nltk.word_tokenize(sentence)
    corrected_tokens = []
    
    for token in tokens:
        # Correct only alphabetic tokens not already in vocabulary
        if token.isalpha():
            if token.lower() in VOCAB_SET:
                corrected_tokens.append(token)
            else:
                corrected = correct_word(token, vocabulary)
                # Preserve original capitalization if token was capitalized
                if token.istitle():
                    corrected = corrected.capitalize()
                corrected_tokens.append(corrected)
        else:
            corrected_tokens.append(token)
            
    # Reconstruct text spacing around punctuation
    return " ".join(corrected_tokens).replace(" ,", ",").replace(" .", ".").replace(" ;", ";").replace(" :", ":").replace(" !", "!").replace(" ?", "?").replace(" '", "'")

print(correct_sentence("This is an exampel of a sentnce with mispelled words."))

# Question 3:
def introduce_spelling_error(word):
    """Inserts a random lowercase letter at a random index in the word."""
    insert_pos = random.randint(0, len(word))
    random_char = random.choice(string.ascii_lowercase)
    return word[:insert_pos] + random_char + word[insert_pos:]

def evaluate_spell_correction(num_sentences=10):
    # Set seed for reproducible evaluation
    random.seed(42)
    
    # 1. Select sentences containing vocabulary words
    raw_sentences = gutenberg.sents('austen-emma.txt')
    # Filter sentences of reasonable length (between 8 and 25 words)
    candidate_sents = [s for s in raw_sentences if 8 <= len(s) <= 25][:num_sentences]
    
    total_altered_words = 0
    correctly_restored_words = 0
    
    print("=" * 80)
    print("EVALUATION & CORRECTION RESULTS")
    print("=" * 80)
    
    for idx, sent_tokens in enumerate(candidate_sents, start=1):
        original_words = []
        altered_tokens = []
        sentence_corrections = []
        
        for token in sent_tokens:
            token_lower = token.lower()
            # 2. Introduce error if word length > 5 and word is in vocabulary
            if token.isalpha() and len(token) > 5 and token_lower in VOCAB_SET:
                corrupted = introduce_spelling_error(token)
                altered_tokens.append(corrupted)
                
                # 3. Correct word using Q1 function
                suggested = correct_word(corrupted)
                
                total_altered_words += 1
                is_correct = (suggested.lower() == token_lower)
                if is_correct:
                    correctly_restored_words += 1
                
                sentence_corrections.append({
                    "original": token,
                    "altered": corrupted,
                    "suggested": suggested,
                    "matched": is_correct
                })
            else:
                altered_tokens.append(token)
        
        # 4. Print results
        original_sentence = " ".join(sent_tokens)
        altered_sentence = " ".join(altered_tokens)
        
        print(f"\n[Sentence {idx}]")
        print(f"Original:  {original_sentence}")
        print(f"Altered:   {altered_sentence}")
        print("Corrections:")
        for item in sentence_corrections:
            status = "MATCH" if item['matched'] else "MISMATCH"
            print(f"  - '{item['altered']}' -> '{item['suggested']}' (Expected: '{item['original']}') [{status}]")

    # 5. Calculate and display precision
    precision = (correctly_restored_words / total_altered_words) if total_altered_words > 0 else 0.0
    print("\n" + "=" * 80)
    print(f"Summary Statistics:")
    print(f"  Total altered words evaluated: {total_altered_words}")
    print(f"  Correctly restored words:     {correctly_restored_words}")
    print(f"  Precision:                     {precision * 100:.2f}%")
    print("=" * 80)

# Run the test pipeline
evaluate_spell_correction(num_sentences=10)

[nltk_data] Downloading package gutenberg to /home/tarek/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /home/tarek/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tarek/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


example
This is an example of a sentence with dispersed words.
EVALUATION & CORRECTION RESULTS

[Sentence 1]
Original:  Between _them_ it was more the intimacy of sisters .
Altered:   Baetween _them_ it was more the intihmacy of siseters .
Corrections:
  - 'Baetween' -> 'between' (Expected: 'Between') [MATCH]
  - 'intihmacy' -> 'intimacy' (Expected: 'intimacy') [MATCH]
  - 'siseters' -> 'sisters' (Expected: 'sisters') [MATCH]

[Sentence 2]
Original:  The danger , however , was at present so unperceived , that they did not by any means rank as misfortunes with her .
Altered:   The dangedr , hsowever , was at presenbt so cunperceived , that they did not by any means rank as mishfortunes with her .
Corrections:
  - 'dangedr' -> 'danger' (Expected: 'danger') [MATCH]
  - 'hsowever' -> 'however' (Expected: 'however') [MATCH]
  - 'presenbt' -> 'present' (Expected: 'present') [MATCH]
  - 'cunperceived' -> 'unperceived' (Expected: 'unperceived') [MATCH]
  - 'mishfortunes' -> 'misfortunes' (Expe

# Exercise 7: Implementing Byte Pair Encoding (BPE) Tokenization

In this exercise, you will implement the Byte Pair Encoding (BPE) tokenization method step by step. Follow the instructions below to build your BPE tokenizer.

### Step 1: Get Vocabulary

Implement the function `get_vocab(sentences)` that takes a list of sentences and returns a dictionary with the counts of each word in the corpus, where each word is written as space-separated characters followed by the end-of-word marker `</w>`.

### Step 2: Get Stats

Implement the function `get_stats(vocab)` that takes the vocabulary dictionary and returns a dictionary of pairs of tokens and their frequencies.

### Step 3: Merge Vocabulary

Implement the function `merge_vocab(vocab, pair)` that takes the vocabulary and a pair of tokens to merge. It should return the updated vocabulary.

### Step 4: BPE

Implement the function `bpe(sentences, num_merges)` that uses the previously defined functions to learn the BPE merges on the provided sentences.

### Step 5: Apply BPE

Implement the function `apply_bpe(sentence, merges)` that takes a sentence and the list of learned merges and returns the tokenized version of the sentence using BPE.

Once you have completed all steps, test your implementation with sample sentences to observe how the BPE tokenization works. You can adjust the number of merges to see how it affects the tokenization.


In [11]:
from collections import defaultdict
import nltk
nltk.download('punkt', quiet=True)      # older NLTK versions
nltk.download('punkt_tab', quiet=True)  # recent NLTK versions


# Step 1: Get Vocabulary
def get_vocab(sentences):
    """
    Functionality:
    -

     '</w>'.
    - '</w>' helps the algorithm distinguish the end of a word from the beginning of the next
      one. It is crucial for keeping word boundaries during the merging process.
    - The vocabulary is a dictionary: key = the word written as space-separated tokens,
      value = how many times the word appears in the corpus.

    Input:
    - sentences: list of strings (sentences or words).
          shape: list of length N, e.g. ["low", "lowest"]

    Output:
    - vocab: dict {str: int}
          shape: one entry per UNIQUE word in the corpus.

    Example:
        sentences = ["low", "lowest"]
        output    = {"l o w </w>": 1, "l o w e s t </w>": 1}

    Hints:
    - Use a defaultdict(int) so you can do vocab[key] += 1 without checking if the key exists.
    - For each sentence: lowercase it, then split it into words with nltk.word_tokenize(...).
    - For each word: list(word) gives its characters; add '</w>' as a last token.
    - Join the tokens with ' ' to build the key, then count it.
    - Return a normal dict (dict(vocab)).
    """
    vocab = defaultdict(int)

    for s in sentences:
        s= nltk.word_tokenize(s.lower())
        for word in s:
            tokens = list(word) + ['</w>']
            key = ' '.join(tokens)
            vocab[key] += 1

    return dict(vocab)


# Step 2: Get Stats
def get_stats(vocab):
    """
    Functionality:
    - Computes the frequency of every pair of ADJACENT tokens (bigram) in the vocabulary.
    - Each pair is counted once per occurrence, weighted by the frequency of the word it
      appears in.
    - '</w>' is treated like any other token.

    Input:
    - vocab: dict {str: int}, keys are space-separated tokens, values are word frequencies.

    Output:
    - pairs: dict {(str, str): int}
          keys   = tuples of two consecutive tokens
          values = total frequency of that pair across the whole vocabulary

    Example:
        vocab = {"l o w </w>": 1, "l o w e s t </w>": 1}
        pairs = {('l','o'): 2, ('o','w'): 2, ('w','</w>'): 1,
                 ('w','e'): 1, ('e','s'): 1, ('s','t'): 1, ('t','</w>'): 1}

    Hints:
    - A word with k tokens has k - 1 adjacent pairs.
    - Use word.split() to get the list of tokens back from the key.
    - Add `freq` (not 1) to the pair count.
    """
    pairs = defaultdict(int)

    for word, freq in vocab.items():
        tokens = word.split()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += freq
    return pairs


# Helper (PROVIDED, do not modify)
def merge_tokens(tokens, pair):
    """
    Merges every adjacent occurrence of `pair` in a LIST of tokens (left to right).

    Input:
    - tokens: list of str, e.g. ['l', 'o', 'w', '</w>']
    - pair:   tuple of two str, e.g. ('o', 'w')

    Output:
    - list of str, e.g. ['l', 'ow', '</w>']

    Why work on a list and not on the raw string?
    With str.replace, the pair ('a', 'b') would wrongly turn "ba b" into "bab".
    Working on tokens avoids merging across token boundaries.
    """
    merged, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged


# Step 3: Merge Vocabulary
def merge_vocab(vocab, pair):
    """
    Functionality:
    - Merges the given pair of tokens into ONE single token in every word of the vocabulary
      and returns the updated vocabulary.

    Input:
    - vocab: dict {str: int}, keys are space-separated tokens, values are word frequencies.
    - pair:  tuple (str, str), the two tokens to merge, e.g. ('o', 'w').

    Output:
    - new_vocab: dict {str: int}, same words but with the pair merged.

    Example:
        old vocab = {"l o w </w>": 1, "l o w e s t </w>": 1}
        pair      = ('o', 'w')
        new vocab = {"l ow </w>": 1, "l ow e s t </w>": 1}

    Hints:
    - Use the provided helper merge_tokens(tokens, pair) on the list word.split().
    - Join the result back with ' ' to obtain the new key.
    - Two different words can become identical after a merge, so ADD the frequencies:
      new_vocab[new_word] = new_vocab.get(new_word, 0) + freq
    """
    new_vocab = {}

    for word, freq in vocab.items():
        words_tokens = word.split()
        merged_tokens = merge_tokens(words_tokens, pair)
        new_word = ' '.join(merged_tokens)
        new_vocab[new_word] = new_vocab.get(new_word, 0) + freq

    return new_vocab


# Step 4: BPE (Byte Pair Encoding)
def bpe(sentences, num_merges):
    """
    Functionality:
    - Learns BPE merges: repeatedly finds the most frequent pair of tokens and merges it,
      num_merges times (or until there is nothing left to merge).

    Input:
    - sentences:  list of str, the training corpus.
    - num_merges: int, the number of merges to perform.

    Output:
    - vocab:  dict {str: int}, the vocabulary after all the merges.
    - merges: list of tuples (str, str), the merged pairs IN THE ORDER they were learned.
              len(merges) <= num_merges.

    Hints:
    - Start from get_vocab(sentences) and an empty list `merges`.
    - At each iteration: pairs = get_stats(vocab).
    - If pairs is empty, stop (break).
    - best_pair = max(pairs, key=pairs.get) gives the most frequent pair.
    - Update vocab with merge_vocab and store best_pair in merges.
    - The ORDER of merges matters: apply_bpe will need it.
    """
    vocab = get_vocab(sentences)
    merges = []

    for _ in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(vocab, best_pair)
        merges.append(best_pair)

    return vocab, merges


# Step 5: Apply BPE
def apply_bpe(sentence, merges):
    """
    Functionality:
    - Tokenizes a NEW sentence using the learned merges, applying each merge in the order in
      which it was learned.

    Input:
    - sentence: str, e.g. "lowering in the fest"
    - merges:   list of tuples (str, str), as returned by bpe().

    Output:
    - list of str: the BPE tokens of the sentence (the '</w>' marker stays in the tokens).

    Hints:
    - Use the SAME pre-processing as in training: lowercase + nltk.word_tokenize.
    - For each word: start from list(word) + ['</w>'].
    - Then, for each pair in merges (in order), tokens = merge_tokens(tokens, pair).
    - Collect the tokens of all the words in a single flat list (extend, not append).
    """
    tokenized_words = []

    words = nltk.word_tokenize(sentence.lower())
    for word in words:
        tokens = list(word) + ['</w>']
        for pair in merges:
            tokens = merge_tokens(tokens, pair)
        tokenized_words.extend(tokens)

    return tokenized_words


# Example Usage / Tests
if __name__ == "__main__":
    # Quick checks: uncomment each one after finishing the corresponding step
    assert get_vocab(["low", "lowest"]) == {"l o w </w>": 1, "l o w e s t </w>": 1}
    assert get_stats({"l o w </w>": 1, "l o w e s t </w>": 1})[('o', 'w')] == 2
    assert merge_vocab({"l o w </w>": 1, "l o w e s t </w>": 1}, ("o", "w")) == \
        {"l ow </w>": 1, "l ow e s t </w>": 1}
    assert merge_vocab({"ba b </w>": 1}, ("a", "b")) == {"ba b </w>": 1}

    # Sample sentences for BPE training
    sentences = [
        "Low",
        "lowest",
        "the",
        "this",
        "fastest"
    ]

    # Perform BPE with a specified number of merges (try different values!)
    num_merges = 10
    vocab, merges = bpe(sentences, num_merges)

    print("Vocabulary after BPE:", vocab)
    print("Learned merges:", merges)

    # Apply BPE to a new sentence that wasn't part of the training sentences
    sentence = "lowering in the fest"
    tokenized_sentence = apply_bpe(sentence, merges)

    print("Tokenized sentence:", tokenized_sentence)

Vocabulary after BPE: {'low</w>': 1, 'lowest</w>': 1, 'the</w>': 1, 'th i s </w>': 1, 'f a st est</w>': 1}
Learned merges: [('s', 't'), ('l', 'o'), ('lo', 'w'), ('e', 'st'), ('est', '</w>'), ('t', 'h'), ('low', '</w>'), ('low', 'est</w>'), ('th', 'e'), ('the', '</w>')]
Tokenized sentence: ['low', 'e', 'r', 'i', 'n', 'g', '</w>', 'i', 'n', '</w>', 'the</w>', 'f', 'est</w>']


# Exercise 8: Using SentencePiece for Byte Pair Encoding (BPE) Tokenization

In this exercise, you will learn how to use the SentencePiece library to implement Byte Pair Encoding (BPE) for tokenization. Follow the steps below to tokenize text using BPE.

### Step 1: Install SentencePiece

Ensure that you have the SentencePiece library installed in your Python environment. You can install it using pip:

```python
!pip install sentencepiece
```

### Step 2: Prepare Your Text
Load and merge all texts from the Gutenberg corpus available in NLTK. You can use the following code snippet to achieve this:

### Step 3: Train the BPE Model
Utilize SentencePiece to train a BPE model on the merged text. Specify the vocabulary size you want to achieve during the training process. **Important:** the default model type of SentencePiece is `unigram`, so you must ask explicitly for `model_type='bpe'`. For example:
```python
import sentencepiece as spm

# Train the SentencePiece model
spm.SentencePieceTrainer.train(input='merged_gutenberg.txt', model_prefix='m',
                               vocab_size=5000, model_type='bpe')
```


### Step 4: Tokenize the Text
Once the model is trained, use it to tokenize your text into subword units. You can test the tokenizer with the following sample sentences:

```python
# Load the trained model
sp = spm.SentencePieceProcessor(model_file='m.model')

# Tokenize a sentence
tokens = sp.encode(sentence, out_type=str)
```

### Step 5: What about Arabic Texts !
Try to do the same thing but this time using an Arabic text corpus.

In [15]:
# Step 1: Install SentencePiece
!pip install sentencepiece

# Step 2: Prepare Your Text
import tokenize

import nltk
from nltk.corpus import gutenberg

# Download the Gutenberg corpus if not already downloaded
nltk.download('gutenberg')

# Get the list of file IDs from the Gutenberg corpus
file_ids = gutenberg.fileids()

# Merge all texts into a single string
merged_text = ""
for file_id in file_ids:
    merged_text += gutenberg.raw(file_id) + "\n"

# Write the merged text to a temporary file
with open('merged_gutenberg.txt', 'w', encoding='utf-8') as f:
    f.write(merged_text)

# step 3: Train the SentencePiece Model
import sentencepiece as spm

# Sample Arabic text
arabic_sample = """
اللغة العربية هي إحدى أكثر اللغات انتشاراً في العالم.
يتحدثها أكثر من 400 مليون نسمة، وتعتبر لغة مقدسة لدى المسلمين.
تتميز بنظام اشتقاقي غني ومعقد يعتمد على الجذور الثلاثية والرباعية.
والعلم نور والجهل ظلام، فاحرص على طلب المعرفة دائماً وأبداً.
"""

# Save to a text file with UTF-8 encoding
with open("arabic_corpus.txt", "w", encoding="utf-8") as f:
    f.write(arabic_sample)
# Train SentencePiece on the Arabic corpus
spm.SentencePieceTrainer.train(
    input="arabic_corpus.txt",
    model_prefix="arabic_spm",
    vocab_size=100,  # Small for demo; use 8000–32000 for large corpora
    model_type="bpe",  # 'bpe' or 'unigram'
    character_coverage=1.0,  # Critical: ensure 100% of Arabic characters are covered
    input_sentence_size=1000000,
    shuffle_input_sentence=True,
)

# Load the trained model
sp_ar = spm.SentencePieceProcessor(model_file="arabic_spm.model")

# Test sentence
sentence = "والعلم نور في الحياة"

# 1. Encode into Subword Tokens (strings)
tokens = sp_ar.encode(sentence, out_type=str)
print("Tokens:")
print(tokens)
# Output will show the subword breakdown with ' ' (U+2581) representing spaces:
# e.g., [' و', 'العلم', ' نور', ' في', ' الح', 'ياة']

# 2. Encode into Token IDs (integers)
token_ids = sp_ar.encode(sentence, out_type=int)
print("\nToken IDs:")
print(token_ids)

# 3. Decode back to the original sentence
decoded_text = sp_ar.decode(tokens)
print("\nDecoded Text:")
print(decoded_text)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Tokens:
['▁وال', 'عل', 'م', '▁ن', 'ور', '▁ف', 'ي', '▁ال', 'ح', 'ي', 'ا', 'ة']

Token IDs:
[11, 7, 66, 23, 20, 22, 67, 4, 82, 67, 64, 72]

Decoded Text:
والعلم نور في الحياة


[nltk_data] Downloading package gutenberg to /home/tarek/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
I0000 00:00:1793009315.501271   74738 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: arabic_corpus.txt
  input_format: 
  model_prefix: arabic_spm
  model_type: BPE
  vocab_size: 100
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 1000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1


## Main differences between our simple BPE implementation and SentencePiece BPE:

### 1. Word-boundary marker (end-of-word vs start-of-word):
- **Simple BPE:** Uses `</w>` at the end of each word.
- **SentencePiece:** Uses `▁` (U+2581 "lower one eighth block", NOT the ASCII underscore `_`) at the beginning of each word.

**Example:**
- Simple BPE: `"low</w> est</w>"`
- SentencePiece: `"▁low ▁est"`

---

### 2. Tokenization approach:
- **Simple BPE:** Pre-tokenizes the text into words (here with NLTK), then applies BPE inside each word.
- **SentencePiece:** Treats the input as a raw stream of characters, spaces included (no external tokenizer is needed, so it is language independent). By default the learned pieces do not cross word boundaries.

**Example (before any merge / after training):**
- Simple BPE: `["l", "o", "w", "</w>", "e", "s", "t", "</w>"]` -> `["low</w>", "est</w>"]`
- SentencePiece: `["▁", "l", "o", "w", "▁", "e", "s", "t"]` -> `["▁low", "▁est"]`

---

### 3. Handling of whitespace:
- **Simple BPE:** Whitespace is discarded during tokenization (only the `</w>` marker remains).
- **SentencePiece:** Whitespace is kept as part of the tokens (it is replaced by `▁`).

**Example:**
- Simple BPE: `"hello world"` -> `["hello</w>", "world</w>"]`
- SentencePiece: `"hello world"` -> `["▁hello", "▁world"]`

---

### 4. Subword regularization:
- **Simple BPE:** No built-in regularization (segmentation is deterministic).
- **SentencePiece:** Supports subword regularization: sampling from the unigram model and BPE-dropout for BPE models.

---

### 5. Unicode handling:
- **Simple BPE:** Python works on Unicode characters, but our pipeline relies on `nltk.word_tokenize`, which is designed for English-like text (it would need another tokenizer for Arabic, Chinese, Japanese...).
- **SentencePiece:** Language independent: works directly on Unicode characters (optionally on bytes with `byte_fallback`), with `character_coverage` controlling which rare characters are kept.

---

### 6. Vocabulary size control:
- **Simple BPE:** Controlled indirectly by the number of merge operations (final vocabulary = initial characters + merges).
- **SentencePiece:** The desired vocabulary size is specified directly (`vocab_size`, which also counts the special tokens `<unk>`, `<s>`, `</s>`).

---

### 7. Out-of-vocabulary (OOV) handling:
- **Simple BPE:** Any word can be split into known characters, but a character never seen in training is simply left as it is (no special token).
- **SentencePiece:** Characters outside the vocabulary are mapped to the special `<unk>` token, unless `byte_fallback` is enabled (bytes are then used instead).

---

### 8. Reversibility (lossless tokenization):
- **Simple BPE:** The output is not a faithful copy of the input: our pipeline lower-cases the text and `word_tokenize` changes quotes, splits punctuation and loses the original spacing, so the exact text cannot be recovered.
- **SentencePiece:** Lossless: whitespace is kept as `▁`, so detokenization is just concatenating the pieces and turning `▁` back into a space (no external detokenizer needed; only the built-in normalization can change the text: Unicode NFKC, and by default removal of leading/trailing/duplicate spaces).

**Example:**
- SentencePiece: `["▁app", "le"]` -> `"▁apple"` -> `"apple"`
- Simple BPE: `["app", "le</w>"]` -> `"apple</w>"` -> `"apple "` (case and spacing of the original text are already lost before BPE even starts)

---

### 9. Efficiency and speed:
- **Simple BPE:** Very slow on large datasets (pure Python, the whole vocabulary is rescanned after every merge).
- **SentencePiece:** Implemented in C++ and highly optimized for speed and efficiency.

### 10. Default model type:
- **SentencePiece** trains a **unigram** language model by default; BPE must be requested with `model_type='bpe'`.